# Data Transformation

Transforms raw landscape data into an `.lcp` file for FARSITE fire spread simulation.

**Pipeline:**
1. Derive slope and aspect from elevation
2. Rasterize canopy attributes from treelist
3. Reproject all layers to EPSG:5070 (Conus Albers, metres)
4. Align all layers to the elevation grid
5. Convert to ASCII rasters
6. Compile `.lcp` file via `lcpmake`
7. Inject projection header into `.lcp`

## Environment Setup

In [ ]:
# Enable reading permissions for executable scripts
!chmod +x install_packages.sh farsite/lcpmake
!./install_packages.sh

In [1]:
import os
import math
import subprocess
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path
from osgeo import gdal, osr
import rasterio
from rasterio.mask import mask

gdal.UseExceptions()
osr.UseExceptions()

# Base directories — resolve() ensures absolute paths throughout
BASE_DIR    = Path(os.path.abspath('')).resolve()
FARSITE_DIR = BASE_DIR / 'farsite'

print(f'Working directory: {BASE_DIR}')

Working directory: /home/jovyan/work/Fire_Risk_Scenario


## Configuration

In [2]:
IGNITION_LAT = 38.9014
IGNITION_LON = -120.0306

# EPSG:5070 (NAD83 / Conus Albers) WKT — used for reprojection
# Written inline to avoid PROJ database lookup issues in this environment
EPSG5070_WKT = 'PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1]]'
WGS84_WKT = 'GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433]]'

## File Paths

In [3]:
# --- executables ---
LCPMAKE_EXEC_PATH = FARSITE_DIR / 'lcpmake'

# --- input files (provided) ---
TIFF_DIR       = BASE_DIR / 'Forest'
ELEV_PATH      = TIFF_DIR / 'Forest_elevation.tif'
FBFM_PATH      = TIFF_DIR / 'Forest_SB40.tif'    # FBFM40
TREELIST_PATH  = TIFF_DIR / 'Forest_Treelist.json'

# --- output directory ---
LCP_OUTPUT_DIR = BASE_DIR / 'Forest_LCP_Outputs'
LCP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- derived rasters (WGS84) ---
SLPP_PATH = LCP_OUTPUT_DIR / 'slope.tif'
ASP_PATH  = LCP_OUTPUT_DIR / 'aspect.tif'
HT_PATH   = LCP_OUTPUT_DIR / 'canopy_height.tif'
CBH_PATH  = LCP_OUTPUT_DIR / 'canopy_base_height.tif'
CBD_PATH  = LCP_OUTPUT_DIR / 'canopy_bulk_density.tif'
CC_PATH   = LCP_OUTPUT_DIR / 'canopy_cover.tif'

# --- reprojected rasters (EPSG:5070, aligned to elevation grid) ---
ELEV_PATH_PROJ = LCP_OUTPUT_DIR / 'elevation_5070.tif'
SLPP_PATH_PROJ = LCP_OUTPUT_DIR / 'slope_5070.tif'
ASP_PATH_PROJ  = LCP_OUTPUT_DIR / 'aspect_5070.tif'
FBFM_PATH_PROJ = LCP_OUTPUT_DIR / 'fbfm40_5070.tif'
HT_PATH_PROJ   = LCP_OUTPUT_DIR / 'canopy_height_5070.tif'
CBH_PATH_PROJ  = LCP_OUTPUT_DIR / 'canopy_base_height_5070.tif'
CBD_PATH_PROJ  = LCP_OUTPUT_DIR / 'canopy_bulk_density_5070.tif'
CC_PATH_PROJ   = LCP_OUTPUT_DIR / 'canopy_cover_5070.tif'

# --- ASCII outputs ---
ELEV_ASCII = LCP_OUTPUT_DIR / 'elevation.asc'
SLPP_ASCII = LCP_OUTPUT_DIR / 'slope.asc'
ASP_ASCII  = LCP_OUTPUT_DIR / 'aspect.asc'
FBFM_ASCII = LCP_OUTPUT_DIR / 'fbfm40.asc'
HT_ASCII   = LCP_OUTPUT_DIR / 'canopy_height.asc'
CBH_ASCII  = LCP_OUTPUT_DIR / 'canopy_base_height.asc'
CBD_ASCII  = LCP_OUTPUT_DIR / 'canopy_bulk_density.asc'
CC_ASCII   = LCP_OUTPUT_DIR / 'canopy_cover.asc'

# --- final LCP ---
LCP_FILEPATH = LCP_OUTPUT_DIR / 'landscape.lcp'

# Validate inputs exist
for p in [ELEV_PATH, FBFM_PATH, TREELIST_PATH, LCPMAKE_EXEC_PATH]:
    status = '✓' if p.exists() else '✗ MISSING'
    print(f'{status}  {p}')

✓  /home/jovyan/work/Fire_Risk_Scenario/Forest/Forest_elevation.tif
✓  /home/jovyan/work/Fire_Risk_Scenario/Forest/Forest_SB40.tif
✓  /home/jovyan/work/Fire_Risk_Scenario/Forest/Forest_Treelist.json
✓  /home/jovyan/work/Fire_Risk_Scenario/farsite/lcpmake


## Step 1 — Derive Slope, Aspect, and Canopy Layers

- Slope and aspect are derived from the elevation TIF.
- Canopy layers (height, base height, bulk density, cover) are rasterized from the treelist JSON.
- All outputs are in WGS84 (degrees), aligned to the elevation grid.

In [4]:
# derive slope and aspect from elevation
gdal.DEMProcessing(str(SLPP_PATH), str(ELEV_PATH), 'slope', slopeFormat='percent')
gdal.DEMProcessing(str(ASP_PATH),  str(ELEV_PATH), 'aspect')
print('✓ slope.tif')
print('✓ aspect.tif')

# read elevation grid properties
# canopy rasters will need to be aligned with the elevation grid
ref_ds = gdal.Open(str(ELEV_PATH))
gt     = ref_ds.GetGeoTransform()
nx     = ref_ds.RasterXSize
ny     = ref_ds.RasterYSize
ref_ds = None

x_min = gt[0]
y_max = gt[3]
x_max = x_min + nx * gt[1]
y_min = y_max + ny * gt[5]

# cell area in meters^2 — needed to calculate canopy cover
# (raster is in degrees so we convert pixel size to meters at the ignition latitude)
lat_rad      = math.radians(IGNITION_LAT)
px_w_m       = abs(gt[1]) * 111320 * math.cos(lat_rad)
px_h_m       = abs(gt[5]) * 111320
cell_area_m2 = px_w_m * px_h_m
print(f'\nElevation grid: {nx}x{ny} cells, ~{px_w_m:.1f}m resolution')
print(f'Extent (WGS84): {x_min:.4f}, {y_min:.4f}, {x_max:.4f}, {y_max:.4f}')

# rasterize canopy height, base height, bulk density from treelist
for out_path, attribute in [
    (HT_PATH,  'HT'),
    (CBH_PATH, 'CBH'),
    (CBD_PATH, 'CBD'),
]:
    gdal.Rasterize(
        str(out_path),
        str(TREELIST_PATH),
        format='GTiff',
        attribute=attribute,
        outputBounds=[x_min, y_min, x_max, y_max],
        width=nx,
        height=ny,
        noData=0,
        outputType=gdal.GDT_Float32,
    )
    print(f'✓ {out_path.name}')

# canopy cover: rasterize crown diameter then convert to cover %
gdal.Rasterize(
    str(CC_PATH),
    str(TREELIST_PATH),
    format='GTiff',
    attribute='DIA',
    outputBounds=[x_min, y_min, x_max, y_max],
    width=nx,
    height=ny,
    noData=0,
    outputType=gdal.GDT_Float32,
)

ds   = gdal.Open(str(CC_PATH), gdal.GA_Update)
band = ds.GetRasterBand(1)
data = band.ReadAsArray().astype(np.float32)
valid       = data != 0
data[valid] = np.clip(
    (np.pi * (data[valid] / 2) ** 2) / cell_area_m2 * 100.0,
    0.0, 100.0
)
band.WriteArray(data)
band.SetNoDataValue(0)
band.FlushCache()
ds = None
print('✓ canopy_cover.tif')

✓ slope.tif
✓ aspect.tif

Elevation grid: 230x180 cells, ~24.1m resolution
Extent (WGS84): -120.0543, 38.8974, -119.9904, 38.9474


Warning 1: PROJ: proj_create_from_database: Open of /opt/conda/share/proj failed
ERROR 1: PROJ: proj_create_from_name: Open of /opt/conda/share/proj failed
Warning 1: PROJ: proj_create_from_database: Open of /opt/conda/share/proj failed
ERROR 1: PROJ: proj_create_from_name: Open of /opt/conda/share/proj failed
ERROR 1: PROJ: proj_create_from_database: Open of /opt/conda/share/proj failed


✓ canopy_height.tif


ERROR 1: PROJ: proj_create_from_database: Open of /opt/conda/share/proj failed


✓ canopy_base_height.tif


ERROR 1: PROJ: proj_create_from_database: Open of /opt/conda/share/proj failed


✓ canopy_bulk_density.tif
✓ canopy_cover.tif


ERROR 1: PROJ: proj_create_from_database: Open of /opt/conda/share/proj failed


## Step 2 — Reproject All Layers to EPSG:5070 and Align to Elevation Grid

FARSITE requires a projected CRS in metres. All layers are reprojected to EPSG:5070
(NAD83 / Conus Albers) and then aligned to exactly match the elevation grid extent
and resolution.

**Key design decisions:**
- FBFM is reprojected directly from the original full-coverage raster — never clipped
  in WGS84 first, as that can shift the ignition point onto non-burnable cells.
- All reprojection uses the `gdalwarp` CLI to avoid PROJ database lookup issues
  in the Python bindings.
- Alignment to the elevation grid happens in projected space (Step 2b), not in
  WGS84, so all extents are consistent before lcpmake runs.

In [5]:
def gdalwarp(src, dst, extra_args=None):
    """Run gdalwarp via CLI. Raises RuntimeError on failure."""
    cmd = ['gdalwarp'] + (extra_args or []) + [str(src), str(dst)]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f'gdalwarp failed for {Path(src).name}:\n{result.stderr}')
    ds = gdal.Open(str(dst))
    print(f'  ✓ {Path(dst).name}: {ds.RasterXSize}x{ds.RasterYSize}  '
          f'unique={np.unique(ds.GetRasterBand(1).ReadAsArray())[:8]}')
    ds = None

# ------------------------------------------------------------------ #
# Step 2a — reproject each original layer independently to EPSG:5070
# ------------------------------------------------------------------ #
print('Reprojecting to EPSG:5070...')
for src, dst, nodata, resample in [
    (ELEV_PATH, ELEV_PATH_PROJ, '-9999', 'bilinear'),
    (SLPP_PATH, SLPP_PATH_PROJ, '-9999', 'bilinear'),
    (ASP_PATH,  ASP_PATH_PROJ,  '-9999', 'bilinear'),
    (FBFM_PATH, FBFM_PATH_PROJ, '-9999', 'near'),     # from ORIGINAL — never pre-clipped
    (HT_PATH,   HT_PATH_PROJ,   '0',     'bilinear'),
    (CBH_PATH,  CBH_PATH_PROJ,  '0',     'bilinear'),
    (CBD_PATH,  CBD_PATH_PROJ,  '0',     'bilinear'),
    (CC_PATH,   CC_PATH_PROJ,   '0',     'bilinear'),
]:
    gdalwarp(src, dst, [
        '-s_srs', WGS84_WKT,
        '-t_srs', EPSG5070_WKT,
        '-r',     resample,
        '-srcnodata', '-9999',
        '-dstnodata', nodata,
        '-of',    'GTiff',
        '-overwrite',
    ])

# ------------------------------------------------------------------ #
# Step 2b — align all layers to exactly match the projected elevation grid
# ------------------------------------------------------------------ #
print('\nAligning to elevation grid...')
ref_ds     = gdal.Open(str(ELEV_PATH_PROJ))
gt_proj    = ref_ds.GetGeoTransform()
nx_proj    = ref_ds.RasterXSize
ny_proj    = ref_ds.RasterYSize
ref_ds     = None

x_min_proj = gt_proj[0]
y_max_proj = gt_proj[3]
x_max_proj = x_min_proj + nx_proj * gt_proj[1]
y_min_proj = y_max_proj + ny_proj * gt_proj[5]

print(f'Master grid (EPSG:5070): {nx_proj}x{ny_proj}  '
      f'extent={x_min_proj:.1f},{y_min_proj:.1f},{x_max_proj:.1f},{y_max_proj:.1f}')

# aligned path variables (overwrite the _5070 paths in place)
for src, dst, nodata, resample in [
    (SLPP_PATH_PROJ, SLPP_PATH_PROJ, '-9999', 'bilinear'),
    (ASP_PATH_PROJ,  ASP_PATH_PROJ,  '-9999', 'bilinear'),
    (FBFM_PATH_PROJ, FBFM_PATH_PROJ, '-9999', 'near'),
    (HT_PATH_PROJ,   HT_PATH_PROJ,   '0',     'bilinear'),
    (CBH_PATH_PROJ,  CBH_PATH_PROJ,  '0',     'bilinear'),
    (CBD_PATH_PROJ,  CBD_PATH_PROJ,  '0',     'bilinear'),
    (CC_PATH_PROJ,   CC_PATH_PROJ,   '0',     'bilinear'),
]:
    tmp = dst.with_stem(dst.stem + '_tmp')
    gdalwarp(src, tmp, [
        '-te',  str(x_min_proj), str(y_min_proj), str(x_max_proj), str(y_max_proj),
        '-ts',  str(nx_proj), str(ny_proj),
        '-r',   resample,
        '-dstnodata', nodata,
        '-of',  'GTiff',
        '-overwrite',
    ])
    tmp.replace(dst)  # atomic rename

print('\nAll layers aligned.')

Reprojecting to EPSG:5070...
  ✓ elevation_5070.tif: 250x253  unique=[-9999.      1866.7196  1866.7478  1868.481   1868.6813  1868.9196
  1868.9419  1868.9656]
  ✓ slope_5070.tif: 250x253  unique=[-9.9990000e+03  9.0595492e-04  9.1086561e-04  9.5557078e-04
  9.8721264e-04  9.9056016e-04  1.0032882e-03  1.0070623e-03]
  ✓ aspect_5070.tif: 250x253  unique=[-9.9990000e+03  2.1990743e+00  2.2646322e+00  2.3261187e+00
  2.4873641e+00  2.5605752e+00  2.5630405e+00  2.6159112e+00]
  ✓ fbfm40_5070.tif: 4939x4955  unique=[-3.4028235e+38 -9.9990000e+03  0.0000000e+00  1.0100000e+02
  1.0200000e+02  1.0300000e+02  1.2100000e+02  1.2200000e+02]
  ✓ canopy_height_5070.tif: 250x253  unique=[0.0000000e+00 1.4012985e-45 1.2872629e-03 2.2299606e-03 3.5581165e-03
 3.8615714e-03 3.8994919e-03 5.4696235e-03]
  ✓ canopy_base_height_5070.tif: 250x253  unique=[0.0000000e+00 1.4012985e-45 7.8124536e-04 9.6496765e-04 1.2443685e-03
 1.4339949e-03 1.9307857e-03 2.4658137e-03]
  ✓ canopy_bulk_density_5070.tif: 25

## Step 3 — Verify Layers Before Building LCP

In [6]:
print('Layer summary (all should match elevation grid size):')
for name, path in [
    ('elevation', ELEV_PATH_PROJ),
    ('slope',     SLPP_PATH_PROJ),
    ('aspect',    ASP_PATH_PROJ),
    ('fuel',      FBFM_PATH_PROJ),
    ('cover',     CC_PATH_PROJ),
    ('height',    HT_PATH_PROJ),
    ('base',      CBH_PATH_PROJ),
    ('density',   CBD_PATH_PROJ),
]:
    ds      = gdal.Open(str(path))
    gt      = ds.GetGeoTransform()
    data    = ds.GetRasterBand(1).ReadAsArray()
    nodata  = ds.GetRasterBand(1).GetNoDataValue()
    valid   = data != nodata if nodata is not None else np.ones_like(data, bool)
    print(f'  {name:12s}: {ds.RasterXSize}x{ds.RasterYSize}  '
          f'res={gt[1]:.1f}m  '
          f'valid={valid.sum():,}  '
          f'unique={np.unique(data[valid])[:6]}')
    ds = None

# check fuel value at ignition point
from pyproj import Transformer
t       = Transformer.from_crs('EPSG:4326', 'EPSG:5070', always_xy=True)
ig_x, ig_y = t.transform(IGNITION_LON, IGNITION_LAT)

ds  = gdal.Open(str(FBFM_PATH_PROJ))
gt  = ds.GetGeoTransform()
col = int((ig_x - gt[0]) / gt[1])
row = int((ig_y - gt[3]) / gt[5])
fuel_val = ds.GetRasterBand(1).ReadAsArray(col, row, 1, 1)[0, 0]
ds  = None

print(f'\nIgnition point EPSG:5070: ({ig_x:.1f}, {ig_y:.1f})')
print(f'Fuel value at ignition:   {fuel_val}  (should be 1-13 or FBFM40 code, not 0/91/98/99)')

Layer summary (all should match elevation grid size):
  elevation   : 250x253  res=26.9m  valid=42,645  unique=[1866.7196 1866.7478 1868.481  1868.6813 1868.9196 1868.9419]
  slope       : 250x253  res=26.9m  valid=41,797  unique=[0.00090595 0.00091087 0.00095557 0.00098721 0.00099056 0.00100329]
  aspect      : 250x253  res=26.9m  valid=41,797  unique=[2.1990743 2.2646322 2.3261187 2.487364  2.5605752 2.5630405]
  fuel        : 250x253  res=26.9m  valid=63,238  unique=[-3.4028235e+38  0.0000000e+00  1.0100000e+02  1.0200000e+02
  1.0300000e+02  1.2100000e+02]
  cover       : 250x253  res=26.9m  valid=42,645  unique=[1.4012985e-45 1.2054972e-04 2.8900852e-04 3.4924460e-04 9.2124718e-04
 1.0205177e-03]
  height      : 250x253  res=26.9m  valid=42,645  unique=[1.4012985e-45 1.2872629e-03 2.2299606e-03 3.5581165e-03 3.8615714e-03
 3.8994919e-03]
  base        : 250x253  res=26.9m  valid=42,645  unique=[1.4012985e-45 7.8124536e-04 9.6496765e-04 1.2443685e-03 1.4339949e-03
 1.9307857e-03]
 

## Step 4 — Convert Projected TIFs to ASCII Rasters

In [7]:
for tif_path, asc_path in [
    (ELEV_PATH_PROJ, ELEV_ASCII),
    (SLPP_PATH_PROJ, SLPP_ASCII),
    (ASP_PATH_PROJ,  ASP_ASCII),
    (FBFM_PATH_PROJ, FBFM_ASCII),
    (HT_PATH_PROJ,   HT_ASCII),
    (CBH_PATH_PROJ,  CBH_ASCII),
    (CBD_PATH_PROJ,  CBD_ASCII),
    (CC_PATH_PROJ,   CC_ASCII),
]:
    gdal.Translate(str(asc_path), str(tif_path), format='AAIGrid')
    print(f'✓ {asc_path.name}')

✓ elevation.asc
✓ slope.asc
✓ aspect.asc
✓ fbfm40.asc
✓ canopy_height.asc
✓ canopy_base_height.asc
✓ canopy_bulk_density.asc
✓ canopy_cover.asc


## Step 5 — Compile `.lcp` File Using `lcpmake`

In [8]:
lcp_stem = LCP_OUTPUT_DIR / 'landscape'

cmd = [
    str(LCPMAKE_EXEC_PATH),
    '-latitude',  str(int(IGNITION_LAT)),
    '-landscape', str(lcp_stem),
    '-elevation', str(ELEV_ASCII),
    '-slope',     str(SLPP_ASCII),
    '-aspect',    str(ASP_ASCII),
    '-fuel',      str(FBFM_ASCII),
    '-cover',     str(CC_ASCII),
    '-height',    str(HT_ASCII),
    '-base',      str(CBH_ASCII),
    '-density',   str(CBD_ASCII),
    '-fbfm40',
]

print('Running lcpmake:')
print(' '.join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True)
print('stdout:', result.stdout)
print('stderr:', result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        f'lcpmake failed (code {result.returncode})\n'
        f'stdout: {result.stdout}\nstderr: {result.stderr}'
    )

print(f'\n✓ Landscape file created: {LCP_FILEPATH}')
print(f'  Size: {LCP_FILEPATH.stat().st_size / 1024:.1f} KB')

Running lcpmake:
/home/jovyan/work/Fire_Risk_Scenario/farsite/lcpmake -latitude 38 -landscape /home/jovyan/work/Fire_Risk_Scenario/Forest_LCP_Outputs/landscape -elevation /home/jovyan/work/Fire_Risk_Scenario/Forest_LCP_Outputs/elevation.asc -slope /home/jovyan/work/Fire_Risk_Scenario/Forest_LCP_Outputs/slope.asc -aspect /home/jovyan/work/Fire_Risk_Scenario/Forest_LCP_Outputs/aspect.asc -fuel /home/jovyan/work/Fire_Risk_Scenario/Forest_LCP_Outputs/fbfm40.asc -cover /home/jovyan/work/Fire_Risk_Scenario/Forest_LCP_Outputs/canopy_cover.asc -height /home/jovyan/work/Fire_Risk_Scenario/Forest_LCP_Outputs/canopy_height.asc -base /home/jovyan/work/Fire_Risk_Scenario/Forest_LCP_Outputs/canopy_base_height.asc -density /home/jovyan/work/Fire_Risk_Scenario/Forest_LCP_Outputs/canopy_bulk_density.asc -fbfm40
stdout: -----------------------------------
-----------------------------------

LCPMake processing grid themes.....

-----------------------------------
-----------------------------------

Siz

## Step 6 — Inject Projection Header into `.lcp`

FARSITE reads CRS info from the LCP binary header. `lcpmake` leaves this field
empty, so we locate it by scanning for the null-padded region after the last
`.asc` path in the header and write the EPSG:5070 WKT directly.

In [9]:
def inject_lcp_projection(lcp_path, wkt_string):
    """Locate the WKT field in the LCP header and write the projection string."""
    with open(lcp_path, 'rb') as f:
        data = f.read()

    # find null region immediately after the last .asc filepath in the header
    last_asc = data.rfind(b'.asc')
    if last_asc == -1:
        raise RuntimeError('Could not find .asc marker in LCP header')

    i = last_asc
    while i < len(data) and data[i] != 0:
        i += 1
    null_start = i

    j = i
    while j < len(data) and data[j] == 0:
        j += 1
    null_length = j - i

    wkt_bytes = wkt_string.encode('utf-8')
    if len(wkt_bytes) > null_length:
        raise ValueError(f'WKT too long: {len(wkt_bytes)} bytes (max {null_length})')

    with open(lcp_path, 'r+b') as f:
        f.seek(null_start)
        f.write(wkt_bytes.ljust(null_length, b'\x00'))

    print(f'✓ Projection written at offset {null_start} ({null_length} bytes available)')

inject_lcp_projection(LCP_FILEPATH, EPSG5070_WKT)

# verify .lcp existence
with open(LCP_FILEPATH, 'rb') as f:
    last_asc_pos = f.read().rfind(b'.asc')

with open(LCP_FILEPATH, 'rb') as f:
    f.seek(last_asc_pos)
    chunk = f.read(512)
null_pos = next(i for i, b in enumerate(chunk) if b == 0)
wkt_start = last_asc_pos + null_pos

with open(LCP_FILEPATH, 'rb') as f:
    f.seek(wkt_start)
    written = f.read(len(EPSG5070_WKT.encode()))
print(f'Verified: {written.decode()[:80]}...')

✓ Projection written at offset 6115 (1201 bytes available)
Verified:                                                                                 ...


## Step 7 — Final Validation

Check that all 8 LCP bands have valid data at the ignition point before running FARSITE.

In [10]:
ds = gdal.Open(str(LCP_FILEPATH))
gt = ds.GetGeoTransform()

col = int((ig_x - gt[0]) / gt[1])
row = int((ig_y - gt[3]) / gt[5])

print(f'LCP size: {ds.RasterXSize} x {ds.RasterYSize}')
print(f'Ignition pixel: row={row}, col={col}')
print(f'In range: col={0 <= col < ds.RasterXSize}, row={0 <= row < ds.RasterYSize}')
print()

non_burnable = {0, 91, 98, 99}
all_ok = True

for band_idx, name in enumerate(['elevation','slope','aspect','fuel','cover','height','base','density'], 1):
    band    = ds.GetRasterBand(band_idx)
    window  = band.ReadAsArray(max(0, col-1), max(0, row-1), 3, 3)
    center  = band.ReadAsArray(col, row, 1, 1)[0, 0]
    if name == 'fuel' and int(center) in non_burnable:
        print(f'  ✗ {name}: {window}  ← NON-BURNABLE, FARSITE will not ignite')
        all_ok = False
    else:
        print(f'  ✓ {name}: center={center}')

ds = None
print()
print('LCP ready for FARSITE.' if all_ok else 'WARNING: fix fuel value before running FARSITE.')

LCP size: 250 x 253
Ignition pixel: row=204, col=77
In range: col=True, row=True

  ✓ elevation: center=2054
  ✓ slope: center=19
  ✓ aspect: center=138
  ✓ fuel: center=144
  ✓ cover: center=1
  ✓ height: center=13
  ✓ base: center=6
  ✓ density: center=0

LCP ready for FARSITE.


# Plot Landscape

In [11]:
def plot_landscape(lcp_output_dir, ignition_lon, ignition_lat):
    """
    Plot all 8 LCP landscape layers in a 2x4 grid with the ignition point overlaid.
    Reads the EPSG:5070 projected GeoTIFFs from the LCP output directory.
    """
    from pyproj import Transformer

    # Transform ignition point to EPSG:5070 for overlay
    t = Transformer.from_crs('EPSG:4326', 'EPSG:5070', always_xy=True)
    ig_x, ig_y = t.transform(ignition_lon, ignition_lat)

    layers = [
        ('elevation_5070.tif',            'Elevation (m)',             'terrain',  None),
        ('slope_5070.tif',                'Slope (%)',                 'YlOrBr',   None),
        ('aspect_5070.tif',               'Aspect (°)',               'hsv',      None),
        ('fbfm40_5070.tif',               'Fuel Model (SB40)',      'Set1',     None),
        ('canopy_cover_5070.tif',         'Canopy Cover (%)',         'Greens',   None),
        ('canopy_height_5070.tif',        'Canopy Height (m)',        'YlGn',     None),
        ('canopy_base_height_5070.tif',   'Canopy Base Height (m)',   'YlGn',     None),
        ('canopy_bulk_density_5070.tif',  'Canopy Bulk Density',      'YlGn',     None),
    ]

    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()

    lcp_dir = Path(lcp_output_dir)

    for ax, (filename, title, cmap, vrange) in zip(axes, layers):
        tif_path = lcp_dir / filename
        ds   = gdal.Open(str(tif_path))
        gt   = ds.GetGeoTransform()
        data = ds.GetRasterBand(1).ReadAsArray().astype(float)
        nodata = ds.GetRasterBand(1).GetNoDataValue()
        ds   = None

        # Mask nodata
        if nodata is not None:
            data[data == nodata] = np.nan

        # For fuel model, also mask non-burnable codes
        if 'Fuel' in title:
            data[np.isin(data.astype(int), [91, 98, 99])] = np.nan

        # Compute extent for plotting with imshow
        x_min = gt[0]
        y_max = gt[3]
        x_max = x_min + data.shape[1] * gt[1]
        y_min = y_max + data.shape[0] * gt[5]
        extent = [x_min, x_max, y_min, y_max]

        im = ax.imshow(data, cmap=cmap, extent=extent, origin='upper',
                        aspect='equal', interpolation='nearest')
        ax.plot(ig_x, ig_y, 'r*', markersize=10, zorder=5)
        ax.set_title(title, fontsize=11, fontweight='bold')
        fig.colorbar(im, ax=ax, shrink=0.7)
        ax.tick_params(labelsize=7)

    fig.suptitle('LCP Landscape Layers (EPSG:5070)', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('farsite_outputs/landscape_layers.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: landscape_layers.png')

In [ ]:
plot_landscape(LCP_OUTPUT_DIR, IGNITION_LON, IGNITION_LAT)

/tmp/ipykernel_10478/1481706822.py:42: RuntimeWarning: invalid value encountered in cast
  data[np.isin(data.astype(int), [91, 98, 99])] = np.nan
